# Split Data Final Clean

Notebook pulito per il workflow attivo di costruzione degli split finali con filtro NLL.

Il flusso esegue:
1. caricamento di `DN`, `DT-`, `DT+`;
2. scelta della reference come prima sequenza di `DN` e calcolo delle distanze;
3. esclusione temporanea delle sequenze `vae` in distanza `[25, 30]`;
4. split train/validation al `15%` sul resto dei dati;
5. reinserimento delle `vae` `[25,30]` nella validation finale;
6. calcolo della NLL sul train con un checkpoint GPT;
7. salvataggio automatico degli output per i percentili NLL richiesti.


## 1) Setup e Utility

In questa sezione importiamo le dipendenze e definiamo le utility minime usate dal flow:
- distanza di Hamming dalla reference;
- scrittura FASTA;
- split train/validation con seed fisso.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from adabmDCA.fasta import get_tokens, import_from_fasta


def hamming_distance_to_reference(seqs: np.ndarray, reference: np.ndarray) -> np.ndarray:
    if seqs.ndim != 2 or reference.ndim != 1:
        raise ValueError("`seqs` deve essere 2D e `reference` 1D")
    if seqs.shape[1] != reference.shape[0]:
        raise ValueError("Lunghezza sequenze e reference non compatibile")
    return (seqs != reference).sum(axis=1).astype(np.int32)


def write_fasta(headers: np.ndarray, seqs: np.ndarray, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    token_list = globals().get("tokens", None)

    with out_path.open("w", encoding="utf-8") as f:
        for header, seq in zip(headers, seqs):
            header_str = str(header)
            if not header_str.startswith(">"):
                header_str = ">" + header_str

            seq_arr = np.asarray(seq)
            if np.issubdtype(seq_arr.dtype, np.integer):
                if token_list is None:
                    raise ValueError("Token RNA non disponibili per decodificare sequenze intere")
                seq_str = "".join(token_list[int(x)] for x in seq_arr.tolist())
            else:
                seq_str = "".join(seq_arr.astype(str).tolist())

            f.write(header_str + "\n")
            f.write(seq_str + "\n")


def split_train_test(headers: np.ndarray, seqs: np.ndarray, test_frac: float = 0.15, seed: int = 42):
    if not (0.0 < test_frac < 1.0):
        raise ValueError("`test_frac` deve essere in (0, 1)")

    n_sequences = seqs.shape[0]
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n_sequences)

    n_test = max(1, int(np.floor(test_frac * n_sequences)))
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return headers[train_idx], seqs[train_idx], headers[test_idx], seqs[test_idx]


## 2) Caricamento Dati e Distanze dalla Reference

Qui carichiamo i tre dataset di partenza, scegliamo la prima sequenza di `DN` come reference e calcoliamo le distanze di Hamming di `DT-` e `DT+` rispetto a quella reference.


In [ ]:
tokens = get_tokens("rna")

headers_DN, data_DN = import_from_fasta(
    "../data/RF00028_aligned_no_inserts_reweighted_d10.fa",
    tokens=tokens,
    filter_sequences=True,
)
headers_DTm, data_DTm = import_from_fasta(
    "../data/DT-_197.fasta",
    tokens=tokens,
    filter_sequences=True,
)
headers_DTp, data_DTp = import_from_fasta(
    "../data/DT+_197.fasta",
    tokens=tokens,
    filter_sequences=True,
)

reference_seq = data_DN[0]
dist_tm = hamming_distance_to_reference(data_DTm, reference_seq)
dist_tp = hamming_distance_to_reference(data_DTp, reference_seq)

print("Shape DN :", data_DN.shape)
print("Shape DT-:", data_DTm.shape)
print("Shape DT+:", data_DTp.shape)
print("Reference length:", reference_seq.shape[0])


## 3) Setup Modello e Helper per la NLL

In questa sezione prepariamo le utility usate per codificare le sequenze, caricare il checkpoint e calcolare la NLL media per token sul train.


In [ ]:
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.dpo_data import _build_labels_from_inputs
from src.transformer import GPTTransformer


def decode_int_encoded_sequences(seqs: np.ndarray, token_list) -> list[str]:
    decoded = []
    for seq in seqs:
        seq_arr = np.asarray(seq)
        if np.issubdtype(seq_arr.dtype, np.integer):
            decoded.append("".join(token_list[int(x)] for x in seq_arr.tolist()))
        else:
            decoded.append("".join(seq_arr.astype(str).tolist()))
    return decoded


def build_stoi_from_dn_fasta(dn_fasta_path: Path):
    with dn_fasta_path.open("r", encoding="utf-8") as f:
        seq_lines = f.read().splitlines()[1::2]
    chars = sorted(list(set("\n".join(seq_lines))))
    pad_symbol = "?"
    chars = chars + [pad_symbol]
    stoi = {ch: i for i, ch in enumerate(chars)}
    return stoi, stoi[pad_symbol]


def encode_with_boundaries(seq: str, stoi: dict[str, int], block_size: int, pad_token: int) -> torch.Tensor:
    token_ids = [stoi[c] for c in f"\n{seq}\n"]
    if len(token_ids) < block_size:
        token_ids += [pad_token] * (block_size - len(token_ids))
    else:
        token_ids = token_ids[:block_size]
    return torch.tensor(token_ids, dtype=torch.long)


def compute_sequence_token_nll(
    model,
    encoded_data: list[torch.Tensor],
    pad_token: int,
    device: torch.device,
    batch_size: int = 256,
) -> np.ndarray:
    nll_batches = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(encoded_data), batch_size):
            x = torch.stack(encoded_data[start : start + batch_size], dim=0).to(device)
            y = _build_labels_from_inputs(x, pad_token)

            logits = model(x)["logits"]
            log_probs = F.log_softmax(logits, dim=-1)
            per_token_logp = torch.gather(log_probs, 2, y.unsqueeze(2)).squeeze(2)
            mask = y != pad_token

            nll_sum = -(per_token_logp * mask).sum(dim=1)
            token_count = mask.sum(dim=1).clamp_min(1)
            nll_batches.append((nll_sum / token_count).detach().cpu().numpy())

    if len(nll_batches) == 0:
        return np.array([], dtype=np.float32)
    return np.concatenate(nll_batches, axis=0)


def to_record_set(headers: np.ndarray, seqs: np.ndarray) -> set[tuple[str, tuple]]:
    records = set()
    for header, seq in zip(headers, seqs):
        seq_arr = np.asarray(seq)
        if np.issubdtype(seq_arr.dtype, np.integer):
            seq_key = tuple(int(x) for x in seq_arr.tolist())
        else:
            seq_key = tuple(seq_arr.astype(str).tolist())
        records.add((str(header), seq_key))
    return records


## 4) Configurazione Manuale del Checkpoint

Qui imposti manualmente il checkpoint da usare per la NLL e le sue caratteristiche architetturali. Il vocabolario puo` essere ricavato automaticamente dal FASTA di `DN`, oppure forzato manualmente.


In [ ]:
project_dir = Path("..").resolve()

MODEL_DN_FASTA_FOR_VOCAB = project_dir / "data/RF00028_aligned_no_inserts_reweighted_d10.fa"
MODEL_CHECKPOINT_PATH = project_dir / "checkpoints/checkpoints_giovanni/RF00028_aligned_GPTTransformer_th10_lr5e-4_batch16_embd64_nhead8_nlayer4_longer_best.pt"
MODEL_VOCAB_SIZE = None
MODEL_PAD_TOKEN_ID = None
MODEL_EMBED_DIM = 64
MODEL_NUM_LAYERS = 4
MODEL_NUM_HEADS = 8
MODEL_MLP_RATIO = 4
MODEL_DROPOUT_P = 0.1
MODEL_MAX_CONTEXT = 198

if not MODEL_DN_FASTA_FOR_VOCAB.exists():
    raise FileNotFoundError(f"DN path per il vocabolario non trovato: {MODEL_DN_FASTA_FOR_VOCAB}")
if not MODEL_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint non trovato: {MODEL_CHECKPOINT_PATH}")

stoi, pad_token_model_auto = build_stoi_from_dn_fasta(MODEL_DN_FASTA_FOR_VOCAB)
vocab_size_model_auto = len(stoi)
vocab_size_model = vocab_size_model_auto if MODEL_VOCAB_SIZE is None else MODEL_VOCAB_SIZE
pad_token_model = pad_token_model_auto if MODEL_PAD_TOKEN_ID is None else MODEL_PAD_TOKEN_ID

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model_nll = GPTTransformer(
    vocab_size=vocab_size_model,
    embed_dim=MODEL_EMBED_DIM,
    num_layers=MODEL_NUM_LAYERS,
    num_heads=MODEL_NUM_HEADS,
    mlp_ratio=MODEL_MLP_RATIO,
    dropout_p=MODEL_DROPOUT_P,
    pad_id=pad_token_model,
)
state_dict = torch.load(MODEL_CHECKPOINT_PATH, map_location="cpu")
model_nll.load_state_dict(state_dict)
model_nll = model_nll.to(device)
model_nll.eval()

print("Model loaded from:", MODEL_CHECKPOINT_PATH)
print("Vocab FASTA:", MODEL_DN_FASTA_FOR_VOCAB)
print("Device:", device)
print("Vocab size:", vocab_size_model, "| auto:", vocab_size_model_auto)
print("Pad token id:", pad_token_model, "| auto:", pad_token_model_auto)
print("Embed dim:", MODEL_EMBED_DIM)
print("Num layers:", MODEL_NUM_LAYERS)
print("Num heads:", MODEL_NUM_HEADS)
print("Max context:", MODEL_MAX_CONTEXT)


## 5) Costruzione dello Split e Calcolo della NLL sul Train

Il flow operativo e` questo:
- identifica le sequenze `vae` a distanza `[25,30]`;
- le esclude temporaneamente;
- fa lo split `85/15` sul resto, separatamente per `DT-` e `DT+`;
- aggiunge le `vae` escluse alla validation finale;
- calcola la NLL sul train risultante.


In [ ]:
seed = 42
test_frac = 0.15
low, high = 25, 30

mask_tm_vae = np.array(["vae" in str(header).lower() for header in headers_DTm], dtype=bool)
mask_tp_vae = np.array(["vae" in str(header).lower() for header in headers_DTp], dtype=bool)

mask_tm_vae_25_30 = mask_tm_vae & (dist_tm >= low) & (dist_tm <= high)
mask_tp_vae_25_30 = mask_tp_vae & (dist_tp >= low) & (dist_tp <= high)

headers_tm_vae_25_30 = headers_DTm[mask_tm_vae_25_30]
data_tm_vae_25_30 = data_DTm[mask_tm_vae_25_30]
headers_tp_vae_25_30 = headers_DTp[mask_tp_vae_25_30]
data_tp_vae_25_30 = data_DTp[mask_tp_vae_25_30]

headers_tm_remaining = headers_DTm[~mask_tm_vae_25_30]
data_tm_remaining = data_DTm[~mask_tm_vae_25_30]
headers_tp_remaining = headers_DTp[~mask_tp_vae_25_30]
data_tp_remaining = data_DTp[~mask_tp_vae_25_30]

if data_tm_remaining.shape[0] == 0 or data_tp_remaining.shape[0] == 0:
    raise ValueError("Dopo esclusione VAE[25,30], DT- o DT+ e' vuoto: impossibile fare split.")

tm_h_tr, tm_x_tr, tm_h_val_split, tm_x_val_split = split_train_test(
    headers_tm_remaining,
    data_tm_remaining,
    test_frac=test_frac,
    seed=seed,
)
tp_h_tr, tp_x_tr, tp_h_val_split, tp_x_val_split = split_train_test(
    headers_tp_remaining,
    data_tp_remaining,
    test_frac=test_frac,
    seed=seed,
)

tm_h_val_final = np.concatenate((tm_h_val_split, headers_tm_vae_25_30), axis=0)
tm_x_val_final = np.concatenate((tm_x_val_split, data_tm_vae_25_30), axis=0)
tp_h_val_final = np.concatenate((tp_h_val_split, headers_tp_vae_25_30), axis=0)
tp_x_val_final = np.concatenate((tp_x_val_split, data_tp_vae_25_30), axis=0)

print("Split base completato")
print(f"DT-: train={tm_x_tr.shape[0]} | val_split={tm_x_val_split.shape[0]} | vae[25,30]={data_tm_vae_25_30.shape[0]} | val_final={tm_x_val_final.shape[0]}")
print(f"DT+: train={tp_x_tr.shape[0]} | val_split={tp_x_val_split.shape[0]} | vae[25,30]={data_tp_vae_25_30.shape[0]} | val_final={tp_x_val_final.shape[0]}")

seqs_tm_train_str = decode_int_encoded_sequences(tm_x_tr, tokens)
seqs_tp_train_str = decode_int_encoded_sequences(tp_x_tr, tokens)

encoded_tm_train = [
    encode_with_boundaries(seq, stoi=stoi, block_size=MODEL_MAX_CONTEXT, pad_token=pad_token_model)
    for seq in seqs_tm_train_str
]
encoded_tp_train = [
    encode_with_boundaries(seq, stoi=stoi, block_size=MODEL_MAX_CONTEXT, pad_token=pad_token_model)
    for seq in seqs_tp_train_str
]

nll_tm_train = compute_sequence_token_nll(
    model_nll,
    encoded_tm_train,
    pad_token=pad_token_model,
    device=device,
    batch_size=256,
)
nll_tp_train = compute_sequence_token_nll(
    model_nll,
    encoded_tp_train,
    pad_token=pad_token_model,
    device=device,
    batch_size=256,
)

dist_tm_train = hamming_distance_to_reference(tm_x_tr, reference_seq)
dist_tp_train = hamming_distance_to_reference(tp_x_tr, reference_seq)

df_train_nll = pd.DataFrame(
    {
        "class": np.concatenate([np.full(len(nll_tm_train), "DT-"), np.full(len(nll_tp_train), "DT+")]),
        "distance": np.concatenate([dist_tm_train, dist_tp_train]).astype(float),
        "nll": np.concatenate([nll_tm_train, nll_tp_train]).astype(float),
    }
)

print("\nNLL train calcolata")
print("DT- train NLL shape:", nll_tm_train.shape)
print("DT+ train NLL shape:", nll_tp_train.shape)
print(df_train_nll.groupby("class")["nll"].describe())


## 6) Percentili NLL e Diagnostica sul Train

I percentili NLL sono configurabili dall'esterno e, di default, sono `70` e `100`.
La soglia viene sempre calcolata sul train `DT-`. `DT-` viene filtrato, mentre `DT+` resta invariato e viene usato solo per controlli diagnostici.


In [ ]:
NLL_THRESHOLD_PERCENTILES = [70, 100]

NLL_THRESHOLD_PERCENTILES = [float(percentile) for percentile in NLL_THRESHOLD_PERCENTILES]
if len(NLL_THRESHOLD_PERCENTILES) == 0:
    raise ValueError("Specifica almeno un percentile in NLL_THRESHOLD_PERCENTILES")
if any((percentile < 0) or (percentile > 100) for percentile in NLL_THRESHOLD_PERCENTILES):
    raise ValueError("I percentili NLL devono essere compresi tra 0 e 100")
if len(set(NLL_THRESHOLD_PERCENTILES)) != len(NLL_THRESHOLD_PERCENTILES):
    raise ValueError("NLL_THRESHOLD_PERCENTILES contiene duplicati")

threshold_results = {}
for percentile in sorted(NLL_THRESHOLD_PERCENTILES):
    threshold = float(np.percentile(nll_tm_train, percentile))
    mask_tm_high_nll = nll_tm_train > threshold
    mask_tp_high_nll = nll_tp_train > threshold
    percentile_tag = str(int(percentile)) if float(percentile).is_integer() else str(percentile).replace(".", "p")

    threshold_results[percentile] = {
        "percentile": percentile,
        "percentile_tag": percentile_tag,
        "threshold": threshold,
        "mask_tm_high_nll": mask_tm_high_nll,
        "mask_tp_high_nll": mask_tp_high_nll,
    }

    print(f"Percentile {percentile_tag} su DT- train -> soglia NLL: {threshold:.6f}")
    print(f"DT- train totali: {len(nll_tm_train)} | sopra soglia: {int(mask_tm_high_nll.sum())} | mantenute: {int((~mask_tm_high_nll).sum())}")
    print(f"DT+ train totali: {len(nll_tp_train)} | sopra soglia: {int(mask_tp_high_nll.sum())} | mantenute nel train finale: {len(nll_tp_train)} (nessun filtro applicato a DT+)")
    print()

fig = plt.figure(figsize=(20, 6), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1.2, 0.8])

ax_scatter = fig.add_subplot(gs[0, 0])
ax_density = fig.add_subplot(gs[0, 1], sharey=ax_scatter)
ax_hist_y = fig.add_subplot(gs[0, 2], sharey=ax_scatter)

ax_scatter.scatter(dist_tm_train, nll_tm_train, s=16, alpha=0.35, c="red", label="DT- train")
ax_scatter.scatter(dist_tp_train, nll_tp_train, s=16, alpha=0.35, c="green", label="DT+ train")
ax_scatter.set_title("TRAIN - NLL vs distanza dalla reference")
ax_scatter.set_xlabel("Distanza di Hamming")
ax_scatter.set_ylabel("NLL media per token")
ax_scatter.legend()
ax_scatter.grid(alpha=0.2)

hb_tm = ax_density.hexbin(dist_tm_train, nll_tm_train, gridsize=28, mincnt=1, cmap="Reds", alpha=0.65)
hb_tp = ax_density.hexbin(dist_tp_train, nll_tp_train, gridsize=28, mincnt=1, cmap="Greens", alpha=0.65)
ax_density.set_title("TRAIN - Densita` punti (DT- rosso, DT+ verde)")
ax_density.set_xlabel("Distanza di Hamming")
ax_density.grid(alpha=0.2)

bins_y = np.linspace(min(nll_tm_train.min(), nll_tp_train.min()), max(nll_tm_train.max(), nll_tp_train.max()), 40)
nll_tp_max = float(nll_tp_train.max())
nll_tp_p99 = float(np.percentile(nll_tp_train, 99.9))
ax_hist_y.hist(nll_tm_train, bins=bins_y, orientation="horizontal", color="red", alpha=0.45, edgecolor="none", label="DT- train")
ax_hist_y.hist(nll_tp_train, bins=bins_y, orientation="horizontal", color="green", alpha=0.35, edgecolor="none", label="DT+ train")
ax_hist_y.set_title("TRAIN - Istogrammi NLL DT-/DT+ (asse Y = NLL)")
ax_hist_y.set_xlabel("Conteggio")
ax_hist_y.legend()
ax_hist_y.grid(alpha=0.2, axis="x")

threshold_colors = ["black", "darkorange", "purple", "brown"]
for idx, result in enumerate(threshold_results.values()):
    color = threshold_colors[idx % len(threshold_colors)]
    threshold = result["threshold"]
    threshold_label = f"p{result['percentile_tag']}={threshold:.3f}"
    for ax in (ax_scatter, ax_density, ax_hist_y):
        ax.axhline(threshold, color=color, linestyle="--", linewidth=2)
    ax_hist_y.text(
        0.98,
        threshold,
        f"  {threshold_label}",
        va="bottom",
        ha="right",
        transform=ax_hist_y.get_yaxis_transform(),
        color=color,
        fontsize=10,
    )

for ax in (ax_scatter, ax_density, ax_hist_y):
    ax.axhline(nll_tp_max, color="green", linestyle=":", linewidth=2)
    ax.axhline(nll_tp_p99, color="blue", linestyle="-.", linewidth=2)

ax_hist_y.text(
    0.98,
    nll_tp_max,
    f"  max DT+ train={nll_tp_max:.3f}",
    va="bottom",
    ha="right",
    transform=ax_hist_y.get_yaxis_transform(),
    color="green",
    fontsize=10,
)
ax_hist_y.text(
    0.98,
    nll_tp_p99,
    f"  p99,9 DT+ train={nll_tp_p99:.3f}",
    va="bottom",
    ha="right",
    transform=ax_hist_y.get_yaxis_transform(),
    color="blue",
    fontsize=10,
)

cbar_tm = fig.colorbar(hb_tm, ax=ax_density, fraction=0.046, pad=0.04)
cbar_tm.set_label("Count DT- train")
cbar_tp = fig.colorbar(hb_tp, ax=ax_density, fraction=0.046, pad=0.12)
cbar_tp.set_label("Count DT+ train")
plt.savefig("nll_vs_distance_train_scatter_density_hist_thresholds.png", dpi=300)
plt.show()


## 7) Salvataggio dei Dataset Finali e Check di Coerenza

Per ogni percentile richiesto:
- `DT- train` viene filtrato con la soglia NLL;
- `DT+ train` viene salvato invariato;
- validation e FASTA di tracciabilita` vengono salvati;
- viene eseguito un controllo esplicito tra i risultati di `70` e `100`.


In [ ]:
saved_outputs = {}

for result in threshold_results.values():
    percentile_tag = result["percentile_tag"]
    threshold = result["threshold"]
    mask_tm_high_nll = result["mask_tm_high_nll"]
    mask_tp_high_nll = result["mask_tp_high_nll"]

    out_dir_section10 = Path(f"../data/split_train_validation/split_0.15_vae25-30-in-val_nll-filtered_{percentile_tag}")
    out_dir_section10.mkdir(parents=True, exist_ok=True)

    tm_h_tr_filtered = tm_h_tr[~mask_tm_high_nll]
    tm_x_tr_filtered = tm_x_tr[~mask_tm_high_nll]
    tp_h_tr_filtered = tp_h_tr.copy()
    tp_x_tr_filtered = tp_x_tr.copy()

    path_tm_train = out_dir_section10 / "DTm_train_after-nll-filter.fasta"
    path_tm_val = out_dir_section10 / "DTm_val_plus-vae25-30.fasta"
    path_tp_train = out_dir_section10 / "DTp_train_after-nll-filter.fasta"
    path_tp_val = out_dir_section10 / "DTp_val_plus-vae25-30.fasta"

    write_fasta(tm_h_tr_filtered, tm_x_tr_filtered, path_tm_train)
    write_fasta(tm_h_val_final, tm_x_val_final, path_tm_val)
    write_fasta(tp_h_tr_filtered, tp_x_tr_filtered, path_tp_train)
    write_fasta(tp_h_val_final, tp_x_val_final, path_tp_val)

    path_tm_removed_nll = out_dir_section10 / "DTm_removed-high-nll-from-train.fasta"
    path_tp_above_nll_diag = out_dir_section10 / "DTp_above-nll-threshold_diagnostic-only.fasta"
    write_fasta(tm_h_tr[mask_tm_high_nll], tm_x_tr[mask_tm_high_nll], path_tm_removed_nll)
    write_fasta(tp_h_tr[mask_tp_high_nll], tp_x_tr[mask_tp_high_nll], path_tp_above_nll_diag)

    path_tm_vae_25_30 = out_dir_section10 / "DTm_vae_25_30_added-to-val.fasta"
    path_tp_vae_25_30 = out_dir_section10 / "DTp_vae_25_30_added-to-val.fasta"
    write_fasta(headers_tm_vae_25_30, data_tm_vae_25_30, path_tm_vae_25_30)
    write_fasta(headers_tp_vae_25_30, data_tp_vae_25_30, path_tp_vae_25_30)

    saved_outputs[percentile_tag] = {
        "out_dir": out_dir_section10,
        "threshold": threshold,
        "tm_train": to_record_set(tm_h_tr_filtered, tm_x_tr_filtered),
        "tp_train": to_record_set(tp_h_tr_filtered, tp_x_tr_filtered),
        "tm_val": to_record_set(tm_h_val_final, tm_x_val_final),
        "tp_val": to_record_set(tp_h_val_final, tp_x_val_final),
        "tm_vae_added": to_record_set(headers_tm_vae_25_30, data_tm_vae_25_30),
        "tp_vae_added": to_record_set(headers_tp_vae_25_30, data_tp_vae_25_30),
        "tm_removed": to_record_set(tm_h_tr[mask_tm_high_nll], tm_x_tr[mask_tm_high_nll]),
        "tp_diag": to_record_set(tp_h_tr[mask_tp_high_nll], tp_x_tr[mask_tp_high_nll]),
    }

    print("Output directory:", out_dir_section10)
    print(f"Percentile NLL: {percentile_tag} | soglia effettiva: {threshold:.6f}")
    print(f"DT- train prima/dopo filtro: {tm_x_tr.shape[0]} / {tm_x_tr_filtered.shape[0]} (rimosse {int(mask_tm_high_nll.sum())})")
    print(f"DT+ train finale: {tp_x_tr.shape[0]} / {tp_x_tr_filtered.shape[0]} (nessuna rimozione; sopra soglia solo diagnostica: {int(mask_tp_high_nll.sum())})")
    print(f"DT- validation finale (split + VAE[25,30]): {tm_x_val_final.shape[0]}")
    print(f"DT+ validation finale (split + VAE[25,30]): {tp_x_val_final.shape[0]}")
    print("File creati:")
    print(path_tm_train)
    print(path_tm_val)
    print(path_tp_train)
    print(path_tp_val)
    print(path_tm_removed_nll)
    print(path_tp_above_nll_diag)
    print(path_tm_vae_25_30)
    print(path_tp_vae_25_30)
    print()

if {"70", "100"}.issubset(saved_outputs):
    tp_subset_ok = saved_outputs["70"]["tp_train"].issubset(saved_outputs["100"]["tp_train"])
    if not tp_subset_ok:
        raise AssertionError("Le sequenze DTp train del percentile 70 non sono un sottoinsieme di quelle del percentile 100")

    invariant_keys = ["tm_val", "tp_val", "tm_vae_added", "tp_vae_added"]
    for key in invariant_keys:
        if saved_outputs["70"][key] != saved_outputs["100"][key]:
            raise AssertionError(f"Mismatch tra percentile 70 e 100 per l'insieme invariato: {key}")

    print("Controlli 70 vs 100 completati")
    print(f"DTp train 70 subset di 100: OK | insiemi uguali: {saved_outputs['70']['tp_train'] == saved_outputs['100']['tp_train']}")
    for key in invariant_keys:
        print(f"{key}: corrispondenza perfetta tra 70 e 100")


## 8) Plot Finale di Conferma

Ultima vista diagnostica: confronto della distribuzione NLL di `DT-` prima/dopo filtro per ciascun percentile richiesto, piu` pannello finale per `DT+`, che resta invariato.


In [ ]:
all_nll_values = np.concatenate([nll_tm_train, nll_tp_train])
bins = np.linspace(float(all_nll_values.min()), float(all_nll_values.max()), 50)

n_percentiles = len(threshold_results)
fig, axes = plt.subplots(1, n_percentiles + 1, figsize=(6 * (n_percentiles + 1), 5), sharey=True)
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, result in zip(axes[:-1], threshold_results.values()):
    nll_tm_kept = nll_tm_train[~result["mask_tm_high_nll"]]
    ax.hist(nll_tm_train, bins=bins, alpha=0.40, color="red", label="DT- train prima", density=True)
    ax.hist(nll_tm_kept, bins=bins, alpha=0.55, color="darkred", label=f"DT- train dopo p{result['percentile_tag']}", density=True)
    ax.axvline(result["threshold"], color="black", linestyle="--", linewidth=2, label=f"soglia p{result['percentile_tag']}")
    ax.set_title(f"DT- train: filtro p{result['percentile_tag']}")
    ax.set_xlabel("NLL media per token")
    ax.set_ylabel("Densita`")
    ax.legend()
    ax.grid(alpha=0.2)

axes[-1].hist(nll_tp_train, bins=bins, alpha=0.40, color="green", label="DT+ train", density=True)
axes[-1].hist(nll_tp_train, bins=bins, alpha=0.55, color="darkgreen", label="DT+ train finale (invariato)", density=True)
for result in threshold_results.values():
    axes[-1].axvline(result["threshold"], color="black", linestyle="--", linewidth=1.5, alpha=0.6)
axes[-1].set_title("DT+ train: nessun filtro applicato")
axes[-1].set_xlabel("NLL media per token")
axes[-1].legend()
axes[-1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Riepilogo filtro NLL su train")
for result in threshold_results.values():
    mask_tm_high_nll = result["mask_tm_high_nll"]
    mask_tp_high_nll = result["mask_tp_high_nll"]
    print(f"p{result['percentile_tag']} | DT- rimosse: {int(mask_tm_high_nll.sum())} / {len(mask_tm_high_nll)}")
    print(f"p{result['percentile_tag']} | DT+ rimosse: 0 / {len(mask_tp_high_nll)} | sopra soglia solo diagnostica: {int(mask_tp_high_nll.sum())}")
